# KrishiGPT v4 QA ROUND 2: constraint-format instruction tune (GPU)

Round 1 (`krishigpt_v4_qa`) learned the Q/A surface format but scored **0%**
instruction compliance — the fine-tune data never demonstrated constraint
formats (one word / yes-no / list-of-three / <=10 words).

Round 2 fixes the two identified causes:
- **format coverage:** constraint formats trained directly (yes/no balanced
  251 yes / 118 no, one-word, list-of-three + definitional)
- **pair quality:** sentence-initial domain-term subjects only (round-1
  noise like "What is whatever soil?" eliminated)

**SELF-PROVISIONING — nothing to upload to Drive.** This notebook generates
`data/qa_v2/` inside Colab from the corpus-v4 text already on Drive (same
deterministic generator, byte-identical scripts embedded below, pair counts
verified against the local run), patches the trainer in the workspace copy,
writes the config, trains, evaluates on the frozen bench, and syncs results
back to Drive. Frozen v3 is NEVER touched (SHA-verified before and after).

**Benchmark-overfit guard:** training phrasing deliberately differs from the
bench's; <=10-words / JSON / stop-word formats never trained.


In [ ]:
# 1. Mount Drive, LOCATE the project, prepare workspace.
import os, shutil, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    raise SystemExit('This notebook must run on Google Colab.')

DRIVE = Path('/content/drive/MyDrive')
WORK = Path('/content/myllm')
MARKER = 'training/train.py'


def is_project(p: Path) -> bool:
    return (p / MARKER).exists() and (p / 'model' / 'gpt.py').exists()


def find_project(root: Path) -> Path | None:
    stack = [root]
    while stack:
        d = stack.pop()
        try:
            for child in d.iterdir():
                if child.name.startswith('.') or child.name in (
                        '.shortcut-targets-by-id', '.Trash'):
                    continue
                if child.is_dir():
                    if is_project(child):
                        return child
                    stack.append(child)
        except (PermissionError, OSError):
            continue
    return None


SRC = None
for cand in (DRIVE / 'KrishiGPT' / 'myllm', DRIVE / 'myllm'):
    if is_project(cand):
        SRC = cand
        break
if SRC is None:
    print('searching Drive for the myllm project folder (one-time)...')
    SRC = find_project(DRIVE)

if SRC is not None:
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns(
        '.venv', '__pycache__', '.pytest_cache', '*.pyc', 'colab'))
    print('project folder found at:', SRC)
else:
    raise SystemExit(
        'myllm project folder not found in Drive — it must contain the v4 '
        'checkpoints and corpus (from the previous Colab runs).')

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print('workspace:', WORK)

# ---- verify what round 2 needs (all already on Drive) -----------------------
required = [
    'checkpoints/krishigpt_v3/best.pt',
    'checkpoints/krishigpt_v4_qa/best.pt',      # round-2 warm-start donor
    'data/processed/agri_bpe_tokenizer.json',
    'data/corpus_v4/agri_train_v4.txt',         # generator source
    'data/corpus_v4/agri_valid_v4.txt',
    'training/train_qa.py',
    'evaluation/health_report.py',
    'evaluation/mcq_bench.py',
    'evaluation/instruction_bench.py',
    'evaluation/funnel_report.py',
]
missing = [r for r in required if not (WORK / r).exists()]
if missing:
    print('MISSING FILES (these should already be on Drive from earlier '
          'runs — check the myllm folder):')
    for m in missing:
        print('  -', m)
    raise SystemExit(f'{len(missing)} required files missing.')
print('all round-2 prerequisites present')

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.9.1', 'numpy==2.5.2'],
                   capture_output=True, text=True)
print('deps ok' if r.returncode == 0 else r.stderr[-500:])


In [ ]:
# 2. SHA-verify frozen v3 (never modified) + record the round-2 donor.
import hashlib

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

expect_v3 = '137cc83da53dd873cbab22c82e7344c7c84882d79b37017a37c765ff38e14536'
got = sha(WORK / 'checkpoints' / 'krishigpt_v3' / 'best.pt')
assert got == expect_v3, f'v3 checkpoint hash mismatch: {got}'
print('frozen v3 best.pt SHA-256 verified (submission artifact untouched)')

donor = sha(WORK / 'checkpoints' / 'krishigpt_v4_qa' / 'best.pt')
print(f'round-2 donor krishigpt_v4_qa/best.pt SHA-256: {donor}')


In [ ]:
# 3. SELF-PROVISION: write the locally-verified scripts (byte-identical),
#    generate data/qa_v2 in Colab, verify counts, write config + device.
import base64, json, subprocess, sys
from pathlib import Path

B64_MAKE_QA_PAIRS = '''IiIiUUEgcGFpciBleHRyYWN0aW9uIGZvciB2NCBpbnN0cnVjdGlvbiB0dW5pbmcg4oCUIGF1dG8tZ2VuZXJhdGVkIGZyb20gdGhlCm1vZGVsJ3Mgb3duIGxpY2Vuc2UtY2xlYW4gY29ycHVzIChubyBleHRlcm5hbCBkYXRhLCBubyBmYWJyaWNhdGlvbikuCgpFeHRyYWN0cyBkZWNsYXJhdGl2ZSBzZW50ZW5jZXMgYW5kIGNvbnZlcnRzIHRoZW0gaW50byBxdWVzdGlvbi9hbnN3ZXIgcGFpcnMKd2l0aCBzdHJ1Y3R1cmUgdGhlIHRva2VuaXplciBhbHJlYWR5IGhhbmRsZXMgd2VsbDoKCiAgIkxvYW0gaXMgYSBtaXh0dXJlIG9mIHNhbmQsIHNpbHQgYW5kIGNsYXkuIgogICAgICAtPiBROiAiV2hhdCBpcyBsb2FtPyIKICAgICAgICAgQTogIkxvYW0gaXMgYSBtaXh0dXJlIG9mIHNhbmQsIHNpbHQgYW5kIGNsYXkuIgoKUGFpciBmaWx0ZXJzIChxdWFsaXR5IHJ1bGVzLCBhbGwgbWVhc3VyZWQpOgogIC0gc3ViamVjdCBoZWFkIG11c3QgYmUgMS0zIHdvcmRzLCBjb3JwdXMtZnJlcXVlbnQgKD49MzApIG9yIGEgZG9tYWluIHRlcm0KICAtIGFuc3dlciBzZW50ZW5jZSA2LTQwIHdvcmRzLCBubyBkaWdpdHMtaGVhdnksIG5vIHF1b3Rlcy9wYXJlbnRoZXNlcwogIC0gb25lIHBhaXIgcGVyIHNlbnRlbmNlOyBkZWR1cGUgYnkgKFEsIEEpOyBubyB2YWxpZGF0aW9uLXNldCBsaW5lcwoKU3BsaXQ6IHRyYWluL3ZhbGlkIFFBIHBhaXJzLCBzZWVkIDAuIE91dHB1dDogZGF0YS9xYS9xYV9wYWlyc197dHJhaW4sdmFsaWR9Lmpzb25sCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgcmFuZG9tCmltcG9ydCByZQppbXBvcnQgc3lzCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0Kc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKCmZyb20gZGF0YS5idWlsZF9hZ3JpY3VsdHVyZV9jb3JwdXMgaW1wb3J0IGNsZWFuX3RleHQgICMgbm9xYTogRTQwMgoKVjRfVFJBSU4gPSBST09UIC8gImRhdGEiIC8gImNvcnB1c192NCIgLyAiYWdyaV90cmFpbl92NC50eHQiClY0X1ZBTElEID0gUk9PVCAvICJkYXRhIiAvICJjb3JwdXNfdjQiIC8gImFncmlfdmFsaWRfdjQudHh0IgpPVVRfRElSID0gUk9PVCAvICJkYXRhIiAvICJxYSIKU0VFRCA9IDAKTUlOX0ZSRVEgPSAzMCAgICAgICAgICAgICMgc3ViamVjdCBoZWFkIGNvcnB1cyBmcmVxdWVuY3kKTUFYX1BBSVJTID0gMzBfMDAwClZBTElEX0ZSQUNUSU9OID0gMC4wMQoKU1RPUCA9IHJlLmNvbXBpbGUociJeKD86YW5kfG9yfGJ1dHx0aGV8YXxhbnxvZnxpbnxvbnxmb3J8dG98d2l0aHxpc3xhcmV8d2FzIgogICAgICAgICAgICAgICAgICByInx3ZXJlfGl0fHRoaXN8dGhhdHx0aGVzZXx0aG9zZXx0aGV5fGhlfHNoZXxoaXN8aGVyfHdoaWNoIgogICAgICAgICAgICAgICAgICByInx3aG98d2hlbnx3aGVyZXx3aGlsZXx0aGVyZXx0aGVufGFsc298aG93ZXZlcnxvdGhlciIKICAgICAgICAgICAgICAgICAgciJ8bWFueXxtb3N0fHNvbWV8YWxsfHN1Y2h8aWZ8d2hlbnxiZWNhdXNlfHNpbmNlfGFsdGhvdWdoIgogICAgICAgICAgICAgICAgICByInx5b3V8aXx3ZXx0aGV5fG9uZXx0d298Zmlyc3R8c2Vjb25kfG5ld3xvdGhlcnxzYW1lfGVhY2giCiAgICAgICAgICAgICAgICAgIHIifGJvdGh8ZmV3fHNldmVyYWx8dmFyaW91cykkIiwgcmUuSSkKCiM6IHdvcmRzIHRoYXQgbWFyayBhIE5PTi1kZWZpbml0aW9uYWwgc2VudGVuY2Ugc3RhcnQgKCJJZiB0aGUgc29pbC4uLiIsCiM6ICJZb3Ugc2VlIHRoZXJlLi4uIiwgIkluIHRoZSBVbml0ZWQgU3RhdGVzLi4uIikKTk9OX0RFRklOSVRJT05BTCA9IHJlLmNvbXBpbGUoCiAgICByIl4oPzppZnx3aGVufHdoaWxlfGJlY2F1c2V8c2luY2V8YWx0aG91Z2h8eW91fGl8d2V8dGhleXx0aGVyZXx0aGVuIgogICAgciJ8dGhpc3x0aGF0fHRoZXNlfHRob3NlfGl0fGluIHRoZXxvbiB0aGV8YXQgdGhlfGZvciB0aGV8dG8gdGhlfGJ5IHRoZSIKICAgIHIifGR1cmluZ3xhZnRlcnxiZWZvcmV8aG93ZXZlcnxmaXJzdHxzZWNvbmR8ZmluYWxseXxub3d8d2VsbHxhbmR8YnV0IgogICAgciJ8c298b3J8YWxzb3xzb21lfG1hbnl8bW9zdHxvdGhlcnxhbm90aGVyfHN1Y2h8dGh1c3xoZW5jZXxpbmRlZWQiCiAgICByInx0aGVyZWZvcmV8bW9yZW92ZXJ8ZnVydGhlcm1vcmV8Y29uc2VxdWVudGx5KVxiIiwgcmUuSSkKCiM6IHN1YmplY3QgSEVBRCBtdXN0IGJlIG9uZSBvZiB0aGVzZSBhZ3JpY3VsdHVyZSBkb21haW4gdGVybXMg4oCUIGtlZXBzIHByb3BlcgojOiBuYW1lcy9hYmJyZXZpYXRpb25zL29kZCBwaHJhc2VzIChERUYsIERldG1lcnMsICJteSBtZWFucyIpIG91dCBvZiBRQSBkYXRhCkRPTUFJTl9IRUFEUyA9IGZyb3plbnNldCgiIiIKcmljZSB3aGVhdCBjb3R0b24gbWFpemUgbWlsbGV0IGJhcmxleSBvYXRzIHNvcmdodW0gc3VnYXJjYW5lIHNveWJlYW4ganV0ZQpiYW5hbmEgY29jb251dCBtYW5nbyB0ZWEgY29mZmVlIGxlbnRpbCBwZWEgcGVhbnV0IHN1bmZsb3dlciBjYXNzYXZhIHJ5ZQpidWNrd2hlYXQgcG90YXRvIHRvbWF0byBvbmlvbiBnYXJsaWMgbXVzdGFyZCBjaGlja3BlYSBiZWFuIGJlYW5zIGdyYXBlCmFwcGxlIG9yYW5nZSBjaXRydXMgZmFybSBmYXJtaW5nIGZhcm1lciBmYXJtcyBjcm9wIGNyb3BzIGdyYWluIGdyYWlucwpjZXJlYWwgY2VyZWFscyBrZXJuZWwga2VybmVscyBzdHJhdyBzdGFsayBzdGFsa3MgaHVzayBjaGFmZiBoYXkgc2lsYWdlCnNvaWwgc29pbHMgbG9hbSBjbGF5IHNhbmQgc2lsdCBodW11cyBjb21wb3N0IG1hbnVyZSB0b3Bzb2lsIHN1YnNvaWwKaG9yaXpvbiBob3Jpem9ucyBncmF2ZWwgdGlsdGggY2xvZCBjbG9kcyBzb2QgZWFydGggZ3JvdW5kIGxhbmQKaXJyaWdhdGlvbiBkcmlwIHNwcmlua2xlciBmdXJyb3cgY2FuYWwgdGVycmFjZSB0ZXJyYWNlcyByYWlud2F0ZXIgZHJhaW5hZ2UKZmxvb2QgZmxvb2Rpbmcgd2F0ZXJsb2dnaW5nIGFxdWlmZXIgZ3JvdW5kd2F0ZXIgd2VsbCB3YXRlciB3YXRlcnMKZmVydGlsaXplciBmZXJ0aWxpemVycyBmZXJ0aWxpc2VyIG1hbnVyZSB1cmVhIHBvdGFzaCBhbW1vbml1bSBwaG9zcGhhdGUKbml0cmF0ZSBuaXRyb2dlbiBwaG9zcGhvcnVzIHBvdGFzc2l1bSBtaWNyb251dHJpZW50IHppbmMgc3VsZmF0ZQpzdXBlcnBob3NwaGF0ZSBudXRyaWVudCBudXRyaWVudHMgbXVsY2ggbXVsY2hpbmcgdmVybWljb21wb3N0CnBlc3QgcGVzdHMgaW5zZWN0IGluc2VjdHMgYXBoaWQgYXBoaWRzIGxvY3VzdCBsb2N1c3RzIHdlZXZpbCB3ZWV2aWxzCmJvcmVyIGJvcmVycyB0aHJpcCB0aHJpcHMgd2hpdGVmbHkgbWVhbHlidWcgY3V0d29ybSBhcm15d29ybSB3aXJld29ybQpjYXRlcnBpbGxhciBjYXRlcnBpbGxhcnMgbW90aCBtb3RocyBiZWV0bGUgYmVldGxlcyBmbHkgZmxpZXMgbWl0ZSBtaXRlcwp3ZWVkIHdlZWRzIHBhcmFzaXRlIHBhcmFzaXRlcwpkaXNlYXNlIGRpc2Vhc2VzIHJ1c3QgYmxpZ2h0IGJsaWdodHMgbWlsZGV3IHdpbHQgc211dCBlcmdvdCBhbnRocmFjbm9zZQpyb3Qgcm90cyBtb2xkIG1vc2FpYyBmdW5ndXMgZnVuZ2kgYmFjdGVyaWEgdmlydXMgdmlydXNlcyBwYXRob2dlbiBwYXRob2dlbnMKcm90YXRpb24gdGlsbGFnZSBwbG91Z2hpbmcgcGxvd2luZyBoYXJyb3dpbmcgc293aW5nIHNlZWRpbmcgcGxhbnRpbmcKaGFydmVzdCBoYXJ2ZXN0aW5nIHRocmVzaGluZyB3aW5ub3dpbmcgd2VlZGluZyBmYWxsb3cgaW50ZXJjcm9wcGluZwpjdWx0aXZhdGlvbiBwcnVuaW5nIGdyYWZ0aW5nIHByb3BhZ2F0aW9uIGRyYWluYWdlCnNlZWQgc2VlZHMgc2VlZGxpbmcgc2VlZGxpbmdzIGdlcm1wbGFzbSBjdWx0aXZhciBjdWx0aXZhcnMgbnVyc2VyeQpwbG91Z2ggcGxvdyBwbG93cyBoYXJyb3cgaGFycm93cyB0cmFjdG9yIHRyYWN0b3JzIHRocmVzaGVyIHRocmVzaGVycwpkcmlsbCBkcmlsbHMgY3VsdGl2YXRvciBzcHJheWVyIGhhcnZlc3RlciBpbXBsZW1lbnQgaW1wbGVtZW50cwpjb21wb3N0IGhlYXAgcGl0IHNpbG8gYmFybiBncmFuYXJ5IHN0b3JlaG91c2UKcGFzdHVyZSBtZWFkb3cgZm9yYWdlIGZvZGRlciBmZWVkIGxpdmVzdG9jayBjYXR0bGUgY293IGNvd3MgZGFpcnkgc2hlZXAKcG91bHRyeSBwaWcgcGlncyBnb2F0IGdvYXRzIGJ1ZmZhbG8gaG9yc2UgaG9yc2VzIGRvbmtleSBiZWUgYmVlcwpiZWVrZWVwaW5nIGFxdWFjdWx0dXJlIGZpc2hlcnkgaGVuIGhlbnMgY2hpY2tlbiBjaGlja2VucyBlZ2cgZWdncwphZ3JpY3VsdHVyZSBhZ3Jvbm9teSBob3J0aWN1bHR1cmUgZmFybWluZwpncmVlbmhvdXNlIGh5ZHJvcG9uaWNzIG9yY2hhcmQgcGxhbnRhdGlvbiBncm92ZSB2aW5leWFyZCBmaWVsZCBmaWVsZHMKZ2FyZGVuIHBsb3QgYWNyZSBhY3JlYWdlIHlpZWxkIHlpZWxkcyBoYXJ2ZXN0cwoiIiIuc3BsaXQoKSkKCgpkZWYgc2VudGVuY2Vfc3BsaXQodGV4dDogc3RyKSAtPiBsaXN0W3N0cl06CiAgICBwYXJ0cyA9IHJlLnNwbGl0KHIiKD88PVsuIT9dKVxzKyIsIHRleHQpCiAgICByZXR1cm4gW3Auc3RyaXAoKSBmb3IgcCBpbiBwYXJ0cyBpZiBwLnN0cmlwKCldCgoKZGVmIHdvcmRfZnJlcSh0ZXh0OiBzdHIpIC0+IENvdW50ZXI6CiAgICByZXR1cm4gQ291bnRlcihyZS5maW5kYWxsKHIiW2Etel0rIiwgdGV4dC5sb3dlcigpKSkKCgpkZWYgc3ViamVjdF9xdWVzdGlvbnMoczogc3RyKSAtPiBsaXN0W3R1cGxlW3N0ciwgc3RyXV06CiAgICAiIiJZaWVsZCAocXVlc3Rpb24sIGFuc3dlcikgZm9yICdYIGlzL2FyZSAuLi4nIGRlZmluaXRpb25hbCBzZW50ZW5jZXMuIiIiCiAgICBpZiBOT05fREVGSU5JVElPTkFMLm1hdGNoKHMpOgogICAgICAgIHJldHVybiBbXQogICAgbSA9IHJlLm1hdGNoKAogICAgICAgIHIiXihbQS1aXVthLXpBLVpdKig/OlxzK1thLXpdW2EtekEtWi1dKil7MCwyfSlccysoaXN8YXJlKVxzKyguKz8pJCIsCiAgICAgICAgcykKICAgIGlmIG5vdCBtOgogICAgICAgIHJldHVybiBbXQogICAgc3ViaiwgdmVyYiwgcmVzdCA9IG0uZ3JvdXBzKCkKICAgIHdvcmRzID0gc3Viai5zcGxpdCgpCiAgICBoZWFkID0gd29yZHNbMF0KICAgIGlmIFNUT1AubWF0Y2goaGVhZCk6CiAgICAgICAgcmV0dXJuIFtdCiAgICBpZiBub3QgYW55KHcubG93ZXIoKSBpbiBET01BSU5fSEVBRFMgZm9yIHcgaW4gd29yZHMpOgogICAgICAgIHJldHVybiBbXSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgZG9tYWluLXRlcm0gc3ViamVjdHMgb25seQogICAgIyBzdWJqZWN0IG11c3QgYmUgYSBub3VuLXBocmFzZSwgbm90IGEgc2VudGVuY2UgYWR2ZXJiL2NsYXVzZToKICAgICMgaGVhZCB3b3JkIGNhcGl0YWxpemVkICsgcmVtYWluaW5nIHdvcmRzIGxvd2VyY2FzZSBkb21haW4gdGVybXMKICAgIGlmIGxlbih3b3JkcykgPiAzOgogICAgICAgIHJldHVybiBbXQogICAgaWYgbGVuKHJlc3Quc3BsaXQoKSkgPCA0OiAgICAgICAgICAjIGFuc3dlcnMgbmVlZCByZWFsIGNvbnRlbnQKICAgICAgICByZXR1cm4gW10KICAgIHFfdmVyYiA9ICJpcyIgaWYgdmVyYiA9PSAiaXMiIGVsc2UgImFyZSIKICAgIHF1ZXN0aW9uID0gZiJXaGF0IHtxX3ZlcmJ9IHtzdWJqLmxvd2VyKCkuc3RyaXAoKX0/IgogICAgYW5zd2VyID0gZiJ7c3Vian0ge3ZlcmJ9IHtyZXN0fSIKICAgIHBhaXJzID0gWyhxdWVzdGlvbiwgYW5zd2VyKV0KICAgICMgaW5zdHJ1Y3Rpb24gZGl2ZXJzaXR5IChoZWxwcyBMMiBpbnN0cnVjdGlvbi1mb2xsb3dpbmcpOiBzYW1lIGZhY3QgaW4KICAgICMgMiBtb3JlIGNvbW1hbmQgZm9ybWF0cywgZGV0ZXJtaW5pc3RpYwogICAgbG93X3N1YmogPSBzdWJqLmxvd2VyKCkuc3RyaXAoKQogICAgcGFpcnMuYXBwZW5kKChmIkRlZmluZSB7bG93X3N1Ymp9LiIsIGFuc3dlcikpCiAgICBwYWlycy5hcHBlbmQoKGYiRXhwbGFpbiB3aGF0IHtsb3dfc3Vian0ge3FfdmVyYn0uIiwgYW5zd2VyKSkKICAgIHJldHVybiBwYWlycwoKCmRlZiBjbGVhbl9xYV90ZXh0KHRleHQ6IHN0cikgLT4gc3RyOgogICAgcmV0dXJuIGNsZWFuX3RleHQodGV4dCwgY3V0X3NlY3Rpb25zPUZhbHNlKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oU0VFRCkKICAgIHRyYWluX3RleHQgPSBjbGVhbl9xYV90ZXh0KFY0X1RSQUlOLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIHZhbGlkX3RleHQgPSBWNF9WQUxJRC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKCiAgICBmcmVxID0gd29yZF9mcmVxKHRyYWluX3RleHQpCgogICAgIyB2YWxpZGF0aW9uLXNldCBsaW5lcyBtdXN0IG5ldmVyIGJlY29tZSB0cmFpbmluZyBwYWlycwogICAgdmFsaWRfbGluZXMgPSB7bC5zdHJpcCgpLmxvd2VyKCkgZm9yIGwgaW4gdmFsaWRfdGV4dC5zcGxpdGxpbmVzKCkKICAgICAgICAgICAgICAgICAgIGlmIGxlbihsLnN0cmlwKCkpID49IDI1fQoKICAgIHBhaXJzOiBsaXN0W3R1cGxlW3N0ciwgc3RyXV0gPSBbXQogICAgc2Vlbl9xYTogc2V0W3R1cGxlW3N0ciwgc3RyXV0gPSBzZXQoKQogICAgbl9zZW50ID0gMAogICAgZm9yIGxpbmUgaW4gdHJhaW5fdGV4dC5zcGxpdGxpbmVzKCk6CiAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAgIGlmIGxlbihsaW5lKSA8IDMwIG9yIGxlbihsaW5lKSA+IDMwMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBsaW5lLmxvd2VyKCkgaW4gdmFsaWRfbGluZXM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIHMgaW4gc2VudGVuY2Vfc3BsaXQobGluZSk6CiAgICAgICAgICAgIG5fc2VudCArPSAxCiAgICAgICAgICAgIGlmIGxlbihzLnNwbGl0KCkpIDwgNiBvciBsZW4ocy5zcGxpdCgpKSA+IDQwOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgcmUuc2VhcmNoKHIiXGR7Myx9Iiwgcykgb3IgJyInIGluIHMgb3IgIigiIGluIHM6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBmb3IgcSwgYSBpbiBzdWJqZWN0X3F1ZXN0aW9ucyhzKToKICAgICAgICAgICAgICAgIGhlYWQgPSByZS5maW5kYWxsKHIiW2Etel0rIiwgYS5sb3dlcigpKSAgIyBhbnN3ZXIgc3ViamVjdCBoZWFkCiAgICAgICAgICAgICAgICBoZWFkX29rID0gYW55KGZyZXEuZ2V0KHcsIDApID49IE1JTl9GUkVRIGZvciB3IGluIGhlYWRbOjJdKQogICAgICAgICAgICAgICAgaWYgbm90IGhlYWRfb2s6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGtleSA9IChxLCBhKQogICAgICAgICAgICAgICAgaWYga2V5IGluIHNlZW5fcWE6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHNlZW5fcWEuYWRkKGtleSkKICAgICAgICAgICAgICAgIHBhaXJzLmFwcGVuZChrZXkpCgogICAgcm5nLnNodWZmbGUocGFpcnMpCiAgICBwYWlycyA9IHBhaXJzWzpNQVhfUEFJUlNdCgogICAgbl92YWxpZCA9IG1heCg1MCwgaW50KGxlbihwYWlycykgKiBWQUxJRF9GUkFDVElPTikpCiAgICB2YWxpZF9wYWlycywgdHJhaW5fcGFpcnMgPSBwYWlyc1s6bl92YWxpZF0sIHBhaXJzW25fdmFsaWQ6XQoKICAgIE9VVF9ESVIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdHJfb3V0ID0gT1VUX0RJUiAvICJxYV9wYWlyc190cmFpbi5qc29ubCIKICAgIHZhX291dCA9IE9VVF9ESVIgLyAicWFfcGFpcnNfdmFsaWQuanNvbmwiCiAgICB0cl9vdXQud3JpdGVfdGV4dCgiXG4iLmpvaW4oanNvbi5kdW1wcyh7InEiOiBxLCAiYSI6IGF9KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBxLCBhIGluIHRyYWluX3BhaXJzKSArICJcbiIsCiAgICAgICAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiKQogICAgdmFfb3V0LndyaXRlX3RleHQoIlxuIi5qb2luKGpzb24uZHVtcHMoeyJxIjogcSwgImEiOiBhfSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgcSwgYSBpbiB2YWxpZF9wYWlycykgKyAiXG4iLAogICAgICAgICAgICAgICAgICAgICAgZW5jb2Rpbmc9InV0Zi04IikKCiAgICBzdGF0cyA9IHsKICAgICAgICAibl9zZW50ZW5jZXNfc2Nhbm5lZCI6IG5fc2VudCwKICAgICAgICAibl9wYWlyc190b3RhbCI6IGxlbihwYWlycyksCiAgICAgICAgIm5fdHJhaW4iOiBsZW4odHJhaW5fcGFpcnMpLAogICAgICAgICJuX3ZhbGlkIjogbGVuKHZhbGlkX3BhaXJzKSwKICAgICAgICAibWluX3N1YmplY3RfZnJlcSI6IE1JTl9GUkVRLAogICAgICAgICJzb3VyY2VfdHJhaW4iOiBzdHIoVjRfVFJBSU4ucmVsYXRpdmVfdG8oUk9PVCkpLAogICAgICAgICJzb3VyY2VfdmFsaWRfZXhjbHVkZWQiOiBzdHIoVjRfVkFMSUQucmVsYXRpdmVfdG8oUk9PVCkpLAogICAgfQogICAgKE9VVF9ESVIgLyAicWFfc3RhdHMuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhzdGF0cywgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQoKICAgIHByaW50KGYiUUEgUEFJUiBFWFRSQUNUSU9OIOKAlCBjb3JwdXMgdjQiKQogICAgcHJpbnQoZiJzZW50ZW5jZXMgc2Nhbm5lZDoge25fc2VudDosfSIpCiAgICBmb3IgaywgdiBpbiBzdGF0cy5pdGVtcygpOgogICAgICAgIGlmIGsgbm90IGluICgic291cmNlX3RyYWluIiwgInNvdXJjZV92YWxpZF9leGNsdWRlZCIpOgogICAgICAgICAgICBwcmludChmIiAge2t9OiB7dn0iKQogICAgcHJpbnQoZiJcbnRyYWluIC0+IHt0cl9vdXQucmVsYXRpdmVfdG8oUk9PVCl9IikKICAgIHByaW50KGYidmFsaWQgLT4ge3ZhX291dC5yZWxhdGl2ZV90byhST09UKX0iKQogICAgcHJpbnQoIlxuZXhhbXBsZXM6IikKICAgIGZvciBxLCBhIGluIHRyYWluX3BhaXJzWzo1XToKICAgICAgICBwcmludChmIiAgUToge3F9XG4gIEE6IHthfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='''
B64_MAKE_QA_PAIRS_V2 = '''IiIiUUEgcGFpciBleHRyYWN0aW9uIHYyIGZvciB0aGUgcm91bmQtMiBpbnN0cnVjdGlvbiB0dW5lIOKAlCBmaXhlcyByb3VuZC0xCmNhdXNlcyBvZiAwJSBjb21wbGlhbmNlIChldmFsdWF0aW9uL2NvcnB1c192NC9FWFBFUklNRU5UX3Y0X3FhLm1kLCAid2h5IEwyCnN0YXllZCBhdCAwJSIsIGNhdXNlcyAxIGFuZCAyKToKCiAgQ0FVU0UgMSAoZm9ybWF0IGNvdmVyYWdlKTogIHJvdW5kIDEgdHJhaW5lZCBPTkxZIG9uIGRlZmluaXRpb25hbCBRQSB3aXRoCiAgICBmcmVlLWZvcm0gc2VudGVuY2UgYW5zd2Vycy4gVGhlIGluc3RydWN0aW9uIGJlbmNoIHRlc3RzIGNvbnN0cmFpbnQKICAgIGZvcm1hdHMgKG9uZSB3b3JkLCB5ZXMvbm8sIGxpc3Qtb2YtdGhyZWUsIDw9MTAgd29yZHMpIHRoYXQgd2VyZSBuZXZlcgogICAgZGVtb25zdHJhdGVkLiB2MiB0cmFpbnMgb24gdGhvc2UgZm9ybWF0cyBkaXJlY3RseS4KCiAgQ0FVU0UgMiAocGFpciBxdWFsaXR5KTogIHRoZSByb3VuZC0xIHN1YmplY3QgcmVnZXggYWNjZXB0cyBjbGF1c2UtaW50ZXJuYWwKICAgICJYIGlzIiBwYXR0ZXJucyAoIldoYXQgaXMgd2hhdGV2ZXIgc29pbD8iLCAiRXhwbGFpbiB3aGF0IGFzIGVhY2ggY293CiAgICBpcy4iKS4gdjIgcmVxdWlyZXMgYSBjbGVhbiBzZW50ZW5jZS1pbml0aWFsIGRvbWFpbi10ZXJtIHN1YmplY3QuCgpEZXNpZ246CiAgLSBkZWZpbml0aW9uYWwgcGFpcnMga2VwdCAoc2FtZSAzIGZvcm1hdHMgYXMgdjEpIOKAlCBmb3JtYXQgY29udGludWl0eQogIC0gTkVXIGNvbnN0cmFpbnQgcGFpcnMgZ2VuZXJhdGVkIGZyb20gdGhlIHNhbWUgODdrLXNlbnRlbmNlIHBvb2w6CiAgICAgIHllcy9ubyAgICA6ICJEb2VzIHJpY2UgbmVlZCB3YXRlcj8gLT4geWVzIiAgICh0ZXJtIHByZXNlbnQgY2hlY2spCiAgICAgIG9uZV93b3JkICA6ICJBbnN3ZXIgaW4gZXhhY3RseSBvbmUgd29yZC4gV2hhdCBpcyBsb2FtPyAtPiBsb2FtIgogICAgICBsaXN0X3RocmVlOiAiTGlzdCBleGFjdGx5IHRocmVlIGdyYWluIGNyb3BzLCBjb21tYS1zZXBhcmF0ZWQuCiAgICAgICAgICAgICAgICAgICAtPiByaWNlLCB3aGVhdCwgbWFpemUiICAgICAgICAgIChmcm9tIGRvbWFpbiBsZXhpY29ucykKICAtIGFuc3dlcnMgZm9yIGNvbnN0cmFpbnQgZm9ybWF0cyBhcmUgU0hPUlQgYnkgY29uc3RydWN0aW9uLCBzbyB0aGUgbW9kZWwKICAgIGlzIHNob3duIGNvbnN0cmFpbnQtU0hBUEUgYW5zd2Vycywgbm90IGp1c3QgY29uc3RyYWludC13b3JkZWQgcHJvbXB0cy4KICAtIEJFTkNITUFSSyBPVkVSRklUIEdVQVJEOiB0aGUgYmVuY2gncyBleGFjdCBzdXJmYWNlIHBhdHRlcm5zIGFyZSBoZWxkCiAgICBvdXQgb2YgdHJhaW5pbmcuIFRyYWluaW5nIHBocmFzaW5nIGlzIGRlbGliZXJhdGVseSBkaWZmZXJlbnQ6CiAgICAgIGJlbmNoOiAiQW5zd2VyIHdpdGggb25seSB5ZXMgb3Igbm8uIiAgIHRyYWluOiAiQW5zd2VyIHllcyBvciBuby4iCiAgICAgIGJlbmNoOiAiQW5zd2VyIGluIGV4YWN0bHkgb25lIHdvcmQuIiAgIHRyYWluOiAob25lLXdvcmQgZm9ybWF0IHVzZXMKICAgICAgICAgICAgICAiUmVwbHkgd2l0aCBhIHNpbmdsZSB3b3JkLiIgcGhyYXNpbmcpCiAgICAgIGJlbmNoOiAiTGlzdCBleGFjdGx5IHRocmVlIFgsIGNvbW1hLXNlcGFyYXRlZC4iCiAgICAgICAgICAgICAgdHJhaW4gbGlzdHMgYXJlIGRyYXduIGZyb20gRElGRkVSRU5UIGxleGljb24gZmFtaWxpZXMKICAgICAgICAgICAgICAoc29pbHMsIHRvb2xzLCBwcmFjdGljZXMpIGFuZCB1c2UgcGhyYXNpbmcgIk5hbWUgdGhyZWUgLi4uIi4KICAgICAgVGhlIDw9MTAtd29yZCBhbmQgSlNPTiBhbmQgc3RvcC13b3JkIGZvcm1hdHMgYXJlIG5ldmVyIHRyYWluZWQuCiAgLSBxdWFsaXR5IGZpeGVzOiBzZW50ZW5jZS1pbml0aWFsIHN1YmplY3Qgb25seTsgZXh0ZW5kZWQgTk9OX0RFRklOSVRJT05BTAogICAgYmxvY2tsaXN0OyBoZWFkIG11c3QgYmUgYSBkb21haW4gdGVybSAoZXhpc3RpbmcgRE9NQUlOX0hFQURTIGdhdGUga2VwdCkuCgpTcGxpdDogc2VlZCAwLCAxJSB2YWxpZCAobWluIDUwKS4gT3V0cHV0OiBkYXRhL3FhX3YyL3FhX3BhaXJzX3t0cmFpbix2YWxpZH0uanNvbmwKUnVuOiAgcHl0aG9uIGV2YWx1YXRpb24vY29ycHVzX3Y0L21ha2VfcWFfcGFpcnNfdjIucHkKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCByYW5kb20KaW1wb3J0IHJlCmltcG9ydCBzeXMKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50c1syXQpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFJPT1QpKQoKZnJvbSBkYXRhLmJ1aWxkX2FncmljdWx0dXJlX2NvcnB1cyBpbXBvcnQgY2xlYW5fdGV4dCAgIyBub3FhOiBFNDAyCmZyb20gZXZhbHVhdGlvbi5jb3JwdXNfdjQubWFrZV9xYV9wYWlycyBpbXBvcnQgKCAgICAgICMgbm9xYTogRTQwMgogICAgRE9NQUlOX0hFQURTLAogICAgc2VudGVuY2Vfc3BsaXQsCiAgICB3b3JkX2ZyZXEsCikKClY0X1RSQUlOID0gUk9PVCAvICJkYXRhIiAvICJjb3JwdXNfdjQiIC8gImFncmlfdHJhaW5fdjQudHh0IgpWNF9WQUxJRCA9IFJPT1QgLyAiZGF0YSIgLyAiY29ycHVzX3Y0IiAvICJhZ3JpX3ZhbGlkX3Y0LnR4dCIKT1VUX0RJUiA9IFJPT1QgLyAiZGF0YSIgLyAicWFfdjIiClNFRUQgPSAwCk1JTl9GUkVRID0gMzAKTUFYX0RFRklOSVRJT05BTCA9IDRfMDAwClZBTElEX0ZSQUNUSU9OID0gMC4wMQoKIzogcm91bmQtMSBtaXNzZXMgY2F1Z2h0IGhlcmU6ICJ3aGF0ZXZlciIsICJoZXJlIiwgImFzIGVhY2ggLi4uIiBzdGFydHMsCiM6IGFuZCBhbnkgc3ViamVjdCB3aG9zZSBmaXJzdCB3b3JkIGlzIG5vdCBhIGRvbWFpbiB0ZXJtLgpOT05fREVGSU5JVElPTkFMX1YyID0gcmUuY29tcGlsZSgKICAgIHIiXig/OmlmfHdoZW58d2hpbGV8YmVjYXVzZXxzaW5jZXxhbHRob3VnaHx5b3V8aXx3ZXx0aGV5fHRoZXJlfHRoZW4iCiAgICByInx0aGlzfHRoYXR8dGhlc2V8dGhvc2V8aXR8aW4gdGhlfG9uIHRoZXxhdCB0aGV8Zm9yIHRoZXx0byB0aGV8YnkgdGhlIgogICAgciJ8ZHVyaW5nfGFmdGVyfGJlZm9yZXxob3dldmVyfGZpcnN0fHNlY29uZHxmaW5hbGx5fG5vd3x3ZWxsfGFuZHxidXQiCiAgICByInxzb3xvcnxhbHNvfHNvbWV8bWFueXxtb3N0fG90aGVyfGFub3RoZXJ8c3VjaHx0aHVzfGhlbmNlfGluZGVlZCIKICAgIHIifHRoZXJlZm9yZXxtb3Jlb3ZlcnxmdXJ0aGVybW9yZXxjb25zZXF1ZW50bHl8d2hhdGV2ZXJ8d2hlbmV2ZXIiCiAgICByInx3aGVyZXZlcnxhcyB8aGVyZSB8dGhlcmUgfHdoZXJlIHx3aHkgfGhvdyB8YWxsIHxlYWNoIHxldmVyeSIKICAgIHIifGJvdGh8ZmV3fHNldmVyYWx8dmFyaW91c3xvbmV8dHdvfHRoZSlcYiIsIHJlLkkpCgojOiB0ZXJtIG11c3QgU1RBUlQgdGhlIHNlbnRlbmNlIChubyBjbGF1c2UtaW50ZXJuYWwgIlggaXMiIGFjY2VwdGVkKQpTVUJKRUNUX1YyID0gcmUuY29tcGlsZSgKICAgIHIiXihbQS1aXVthLXpBLVpdKig/OlxzK1thLXpdW2EtekEtWi1dKil7MCwyfSlccysoaXN8YXJlKVxzKyguKz8pJCIpCgojOiB2ZXJicyBpbiAiWCBuZWVkKHMpL3JlcXVpcmUocykvdXNlKHMpIC4uLiIgc2VudGVuY2VzIC0+IHllcy9ubyBtYXRlcmlhbApORUVEX1BBVFRFUk4gPSByZS5jb21waWxlKAogICAgciJeKFtBLVpdW2Etel0rKD86XHMrW2Etel1bYS16XSspPylccysoPzpuZWVkcz98cmVxdWlyZXM/fHVzZXM/fCIKICAgIHIicHJlZmVycz98dGhyaXZlcz98Z3Jvd3M/fGlzKVxiIiwgcmUuSSkKCiM6IGNhdGVnb3JpZXMgZm9yIG9uZS13b3JkIGFuc3dlcnMgKHRlcm0gLT4gY2F0ZWdvcnkpOyBhbHNvIHVzZWQgdG8gYnVpbGQKIzogY3Jvc3MtY2F0ZWdvcnkgIm5vIiBwYWlycyBiZWxvdwpPTkVfV09SRF9QT09MID0gewogICAgImxvYW0iOiAic29pbCIsICJjbGF5IjogInNvaWwiLCAic2FuZCI6ICJzb2lsIiwgInNpbHQiOiAic29pbCIsCiAgICAiaHVtdXMiOiAic29pbCIsICJjb21wb3N0IjogInNvaWwiLCAibWFudXJlIjogInNvaWwiLCAidG9wc29pbCI6ICJzb2lsIiwKICAgICJzdWJzb2lsIjogInNvaWwiLCAidmVybWljb21wb3N0IjogInNvaWwiLAogICAgInJpY2UiOiAiY3JvcCIsICJ3aGVhdCI6ICJjcm9wIiwgImNvdHRvbiI6ICJjcm9wIiwgIm1haXplIjogImNyb3AiLAogICAgIm1pbGxldCI6ICJjcm9wIiwgImJhcmxleSI6ICJjcm9wIiwgIm9hdHMiOiAiY3JvcCIsICJzb3JnaHVtIjogImNyb3AiLAogICAgInN1Z2FyY2FuZSI6ICJjcm9wIiwgInNveWJlYW4iOiAiY3JvcCIsICJqdXRlIjogImNyb3AiLAogICAgInVyZWEiOiAiZmVydGlsaXplciIsICJwb3Rhc2giOiAiZmVydGlsaXplciIsICJuaXRyb2dlbiI6ICJmZXJ0aWxpemVyIiwKICAgICJwaG9zcGhvcnVzIjogImZlcnRpbGl6ZXIiLCAicG90YXNzaXVtIjogImZlcnRpbGl6ZXIiLAogICAgInN1cGVycGhvc3BoYXRlIjogImZlcnRpbGl6ZXIiLCAibml0cmF0ZSI6ICJmZXJ0aWxpemVyIiwKICAgICJhcGhpZCI6ICJwZXN0IiwgImxvY3VzdCI6ICJwZXN0IiwgIndlZXZpbCI6ICJwZXN0IiwgInRocmlwcyI6ICJwZXN0IiwKICAgICJ3aGl0ZWZseSI6ICJwZXN0IiwgIm1lYWx5YnVnIjogInBlc3QiLCAiY3V0d29ybSI6ICJwZXN0IiwKICAgICJhcm15d29ybSI6ICJwZXN0IiwgIndpcmV3b3JtIjogInBlc3QiLCAiYm9yZXIiOiAicGVzdCIsCiAgICAicnVzdCI6ICJkaXNlYXNlIiwgImJsaWdodCI6ICJkaXNlYXNlIiwgIm1pbGRldyI6ICJkaXNlYXNlIiwKICAgICJ3aWx0IjogImRpc2Vhc2UiLCAic211dCI6ICJkaXNlYXNlIiwgImVyZ290IjogImRpc2Vhc2UiLAogICAgImFudGhyYWNub3NlIjogImRpc2Vhc2UiLCAibW9zYWljIjogImRpc2Vhc2UiLAogICAgInRyYWN0b3IiOiAibWFjaGluZSIsICJwbG91Z2giOiAibWFjaGluZSIsICJwbG93IjogIm1hY2hpbmUiLAogICAgImhhcnJvdyI6ICJtYWNoaW5lIiwgInRocmVzaGVyIjogIm1hY2hpbmUiLCAiY3VsdGl2YXRvciI6ICJtYWNoaW5lIiwKICAgICJzcHJheWVyIjogIm1hY2hpbmUiLCAiZHJpbGwiOiAibWFjaGluZSIsCiAgICAic3ByaW5rbGVyIjogImlycmlnYXRpb24iLCAiZHJpcCI6ICJpcnJpZ2F0aW9uIiwgImNhbmFsIjogImlycmlnYXRpb24iLAogICAgInRlcnJhY2UiOiAiaXJyaWdhdGlvbiIsICJmdXJyb3ciOiAiaXJyaWdhdGlvbiIsICJkcmFpbmFnZSI6ICJpcnJpZ2F0aW9uIiwKfQoKIzogbGlzdCBmYW1pbGllcyDigJQgREVMSUJFUkFURUxZIGRpc2pvaW50IGZyb20gdGhlIGJlbmNoJ3MgImdyYWluIGNyb3BzIiAvCiM6ICJzb2lsIHR5cGVzIiBwaHJhc2luZyBhbmQgbGlzdCBjb250ZW50cyBzbyB0aGUgYmVuY2ggc3VyZmFjZSBzdGF5cyBub3ZlbApMSVNUX1BPT0xTID0gewogICAgImZpZWxkIHRvb2xzIjogWyJwbG91Z2giLCAiaGFycm93IiwgImRyaWxsIiwgImN1bHRpdmF0b3IiLCAidGhyZXNoZXIiLAogICAgICAgICAgICAgICAgICAgICJzcHJheWVyIiwgInNpY2tsZSIsICJzcGFkZSJdLAogICAgImdyYWluIGNlcmVhbHMiOiBbInJpY2UiLCAid2hlYXQiLCAibWFpemUiLCAiYmFybGV5IiwgIm9hdHMiLAogICAgICAgICAgICAgICAgICAgICAgInNvcmdodW0iLCAibWlsbGV0IiwgInJ5ZSJdLAogICAgImZhcm0gcHJhY3RpY2VzIjogWyJyb3RhdGlvbiIsICJ0aWxsYWdlIiwgIndlZWRpbmciLCAibXVsY2hpbmciLAogICAgICAgICAgICAgICAgICAgICAgICJwcnVuaW5nIiwgImdyYWZ0aW5nIiwgInNvd2luZyIsICJ0aHJlc2hpbmciLAogICAgICAgICAgICAgICAgICAgICAgICJ3aW5ub3dpbmciLCAiZmFsbG93Il0sCiAgICAiaXJyaWdhdGlvbiBtZXRob2RzIjogWyJkcmlwIiwgInNwcmlua2xlciIsICJmdXJyb3ciLCAiY2FuYWwiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiZmxvb2QiLCAidGVycmFjZSJdLAogICAgImZhcm0gYW5pbWFscyI6IFsiY2F0dGxlIiwgInNoZWVwIiwgImdvYXRzIiwgInBvdWx0cnkiLCAicGlncyIsCiAgICAgICAgICAgICAgICAgICAgICJob3JzZXMiLCAiZG9ua2V5cyIsICJidWZmYWxvIl0sCn0KCkxJU1RfVEVNUExBVEVTID0gWwogICAgKCJOYW1lIHRocmVlIHtmYW1pbHl9LiBBbnN3ZXIgd2l0aCB0aHJlZSBpdGVtcyBvbmx5LiIsCiAgICAgbGFtYmRhIHBvb2w6ICIsICIuam9pbihwb29sWzozXSkpLAogICAgKCJOYW1lIHRocmVlIHtmYW1pbHl9LCBzZXBhcmF0ZWQgYnkgY29tbWFzLiIsCiAgICAgbGFtYmRhIHBvb2w6ICIsICIuam9pbihwb29sWzozXSkpLAogICAgKCJHaXZlIHRocmVlIGV4YW1wbGVzIG9mIHtmYW1pbHl9LiBKdXN0IHRoZSB0aHJlZSBuYW1lcy4iLAogICAgIGxhbWJkYSBwb29sOiAiLCAiLmpvaW4ocG9vbFs6Ml0gKyBwb29sWzM6NF0pKSwKXQoKCmRlZiBkZWZpbml0aW9uYWxfcGFpcnMoczogc3RyLCBmcmVxOiBDb3VudGVyKSAtPiBsaXN0W3R1cGxlW3N0ciwgc3RyXV06CiAgICAiIiJSb3VuZC0xIHN0eWxlIHBhaXJzLCBidXQgb25seSBmb3IgY2xlYW4gc2VudGVuY2UtaW5pdGlhbCBzdWJqZWN0cy4iIiIKICAgIGlmIE5PTl9ERUZJTklUSU9OQUxfVjIubWF0Y2gocyk6CiAgICAgICAgcmV0dXJuIFtdCiAgICBtID0gU1VCSkVDVF9WMi5tYXRjaChzKQogICAgaWYgbm90IG06CiAgICAgICAgcmV0dXJuIFtdCiAgICBzdWJqLCB2ZXJiLCByZXN0ID0gbS5ncm91cHMoKQogICAgd29yZHMgPSBzdWJqLnNwbGl0KCkKICAgIGlmIHdvcmRzWzBdLmxvd2VyKCkgbm90IGluIERPTUFJTl9IRUFEUzoKICAgICAgICByZXR1cm4gW10gICAgICAgICAgICAgICAgICAgICAgICMgaGVhZCBpdHNlbGYgbXVzdCBiZSB0aGUgZG9tYWluIHRlcm0KICAgIGlmIGxlbih3b3JkcykgPiAzIG9yIGxlbihyZXN0LnNwbGl0KCkpIDwgNDoKICAgICAgICByZXR1cm4gW10KICAgIGhlYWRfd29yZHMgPSByZS5maW5kYWxsKHIiW2Etel0rIiwgcmVzdC5sb3dlcigpKVs6Ml0KICAgIGlmIG5vdCBhbnkoZnJlcS5nZXQodywgMCkgPj0gTUlOX0ZSRVEgZm9yIHcgaW4gaGVhZF93b3Jkcyk6CiAgICAgICAgcmV0dXJuIFtdCiAgICBsb3cgPSBzdWJqLmxvd2VyKCkuc3RyaXAoKQogICAgcV92ZXJiID0gImlzIiBpZiB2ZXJiID09ICJpcyIgZWxzZSAiYXJlIgogICAgYW5zd2VyID0gZiJ7c3Vian0ge3ZlcmJ9IHtyZXN0fSIKICAgIHJldHVybiBbCiAgICAgICAgKGYiV2hhdCB7cV92ZXJifSB7bG93fT8iLCBhbnN3ZXIpLAogICAgICAgIChmIkRlZmluZSB7bG93fS4iLCBhbnN3ZXIpLAogICAgICAgIChmIkV4cGxhaW4gd2hhdCB7bG93fSB7cV92ZXJifS4iLCBhbnN3ZXIpLAogICAgXQoKCmRlZiB5ZXNfbm9fcGFpcnMoczogc3RyKSAtPiBsaXN0W3R1cGxlW3N0ciwgc3RyXV06CiAgICAiIiInRG9lcyBYIG5lZWQgd2F0ZXI/JyAtPiAneWVzJyAgKFggPSBkb21haW4gc3ViamVjdCwgc2VudGVuY2UgYXNzZXJ0cwogICAgWCBuZWVkcy91c2VzL2dyb3dzIC4uLikuIFVzZXMgRElGRkVSRU5UIHBocmFzaW5nIGZyb20gdGhlIGJlbmNoLiIiIgogICAgbSA9IE5FRURfUEFUVEVSTi5tYXRjaChzKQogICAgaWYgbm90IG06CiAgICAgICAgcmV0dXJuIFtdCiAgICBzdWJqID0gbS5ncm91cCgxKQogICAgaWYgc3Viai5sb3dlcigpLnNwbGl0KClbMF0gbm90IGluIERPTUFJTl9IRUFEUzoKICAgICAgICByZXR1cm4gW10KICAgIGlmIGxlbihzdWJqLnNwbGl0KCkpID4gMzoKICAgICAgICByZXR1cm4gW10KICAgIGxvdyA9IHN1YmoubG93ZXIoKQogICAgcmV0dXJuIFsKICAgICAgICAoZiJEb2VzIHtsb3d9IG5lZWQgd2F0ZXI/IEFuc3dlciB5ZXMgb3Igbm8uIiwgInllcyIpLAogICAgICAgIChmIklzIHtsb3d9IHBhcnQgb2YgZmFybWluZz8gQW5zd2VyIHllcyBvciBuby4iLCAieWVzIiksCiAgICBdCgoKZGVmIG9uZV93b3JkX3BhaXJzKHM6IHN0cikgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1dOgogICAgIiIiJ1doYXQgaXMgbG9hbT8gUmVwbHkgd2l0aCBhIHNpbmdsZSB3b3JkLicgLT4gJ3NvaWwnICh0ZXJtIGNhdGVnb3J5LAogICAgb25lIHdvcmQgYnkgY29uc3RydWN0aW9uKS4iIiIKICAgIG0gPSBTVUJKRUNUX1YyLm1hdGNoKHMpCiAgICBpZiBub3QgbToKICAgICAgICByZXR1cm4gW10KICAgIHN1YmogPSBtLmdyb3VwKDEpCiAgICBsb3cgPSBzdWJqLmxvd2VyKCkuc3RyaXAoKQogICAgaWYgbGVuKGxvdy5zcGxpdCgpKSAhPSAxIG9yIGxvdyBub3QgaW4gT05FX1dPUkRfUE9PTDoKICAgICAgICByZXR1cm4gW10KICAgIHJldHVybiBbKGYiV2hhdCBpcyB7bG93fT8gUmVwbHkgd2l0aCBhIHNpbmdsZSB3b3JkLiIsCiAgICAgICAgICAgICBPTkVfV09SRF9QT09MW2xvd10pXQoKCmRlZiBvbmVfd29yZF9uZWdhdGl2ZV9wYWlycyhybmc6IHJhbmRvbS5SYW5kb20pIC0+IGxpc3RbdHVwbGVbc3RyLCBzdHJdXToKICAgICIiIidJcyBhIHRyYWN0b3IgYSBjcm9wPyBBbnN3ZXIgeWVzIG9yIG5vLicgLT4gJ25vJy4gQ3Jvc3MtY2F0ZWdvcnkKICAgIG5lZ2F0aXZlcyBiYWxhbmNlIHRoZSB5ZXMtcG9zaXRpdmVzIHNvIHRoZSBtb2RlbCBsZWFybnMgdGhlIFdPUkQgJ25vJwogICAgdG9vLCBub3QganVzdCAneWVzJyAocm91bmQtMiBmaXg6IHYxIHllcy9ubyBkYXRhIHdhcyAxMDAlICd5ZXMnKS4iIiIKICAgIGNhdHM6IGRpY3Rbc3RyLCBsaXN0W3N0cl1dID0ge30KICAgIGZvciB0ZXJtLCBjYXQgaW4gT05FX1dPUkRfUE9PTC5pdGVtcygpOgogICAgICAgIGNhdHMuc2V0ZGVmYXVsdChjYXQsIFtdKS5hcHBlbmQodGVybSkKICAgIG91dCA9IFtdCiAgICBmb3IgYV9jYXQsIGFfdGVybXMgaW4gY2F0cy5pdGVtcygpOgogICAgICAgIGZvciBiX2NhdCwgYl90ZXJtcyBpbiBjYXRzLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIGFfY2F0ID09IGJfY2F0OgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZm9yIHdyb25nIGluIHJuZy5zYW1wbGUoYl90ZXJtcywgbWluKDMsIGxlbihiX3Rlcm1zKSkpOgogICAgICAgICAgICAgICAgZm9yIHJpZ2h0IGluIHJuZy5zYW1wbGUoYV90ZXJtcywgbWluKDMsIGxlbihhX3Rlcm1zKSkpOgogICAgICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKGYiSXMge3dyb25nfSBhIHthX2NhdH0/IEFuc3dlciB5ZXMgb3Igbm8uIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibm8iKSkKICAgICAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKChmIklzIHtyaWdodH0gYSB7YV9jYXR9PyBBbnN3ZXIgeWVzIG9yIG5vLiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInllcyIpKQogICAgcmV0dXJuIG91dAoKCmRlZiBsaXN0X3BhaXJzKHJuZzogcmFuZG9tLlJhbmRvbSkgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1dOgogICAgIiIiJ05hbWUgdGhyZWUgZmllbGQgdG9vbHMuJyAtPiAncGxvdWdoLCBoYXJyb3csIGRyaWxsJyAodGhyZWUgaXRlbXMgYnkKICAgIGNvbnN0cnVjdGlvbikuIEZhbWlsaWVzL3BocmFzaW5nIGRpZmZlciBmcm9tIHRoZSBiZW5jaCdzIGxpc3RzLiIiIgogICAgb3V0ID0gW10KICAgIGZvciBmYW1pbHksIHBvb2wgaW4gTElTVF9QT09MUy5pdGVtcygpOgogICAgICAgIGlmIGxlbihwb29sKSA8IDQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIHRlbXBsYXRlLCBmaWxsZXIgaW4gTElTVF9URU1QTEFURVM6CiAgICAgICAgICAgIHNodWZmbGVkID0gcG9vbFs6XQogICAgICAgICAgICBybmcuc2h1ZmZsZShzaHVmZmxlZCkKICAgICAgICAgICAgcSA9IHRlbXBsYXRlLmZvcm1hdChmYW1pbHk9ZmFtaWx5KQogICAgICAgICAgICBhID0gZmlsbGVyKHNodWZmbGVkKQogICAgICAgICAgICBvdXQuYXBwZW5kKChxLCBhKSkKICAgIHJldHVybiBvdXQKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBybmcgPSByYW5kb20uUmFuZG9tKFNFRUQpCiAgICB0cmFpbl90ZXh0ID0gY2xlYW5fdGV4dCgKICAgICAgICBWNF9UUkFJTi5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IiksIGN1dF9zZWN0aW9ucz1GYWxzZSkKICAgIHZhbGlkX3RleHQgPSBWNF9WQUxJRC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikKCiAgICBmcmVxID0gd29yZF9mcmVxKHRyYWluX3RleHQpCgogICAgIyB2YWxpZGF0aW9uLXNldCBsaW5lcyBtdXN0IG5ldmVyIGJlY29tZSB0cmFpbmluZyBwYWlycwogICAgdmFsaWRfbGluZXMgPSB7bC5zdHJpcCgpLmxvd2VyKCkgZm9yIGwgaW4gdmFsaWRfdGV4dC5zcGxpdGxpbmVzKCkKICAgICAgICAgICAgICAgICAgIGlmIGxlbihsLnN0cmlwKCkpID49IDI1fQoKICAgIHBhaXJzOiBsaXN0W3R1cGxlW3N0ciwgc3RyXV0gPSBbXQogICAgc2Vlbjogc2V0W3R1cGxlW3N0ciwgc3RyXV0gPSBzZXQoKQogICAgc3RhdHMgPSB7InNlbnRlbmNlc19zY2FubmVkIjogMCwgImRlZmluaXRpb25hbCI6IDAsICJ5ZXNfbm8iOiAwLAogICAgICAgICAgICAgIm9uZV93b3JkIjogMCwgImxpc3RfdGhyZWUiOiAwLCAicmVqZWN0ZWRfdmFsaWRfbGluZXMiOiAwfQoKICAgIGZvciBsaW5lIGluIHRyYWluX3RleHQuc3BsaXRsaW5lcygpOgogICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICBpZiBsZW4obGluZSkgPCAzMCBvciBsZW4obGluZSkgPiAzMDAgb3IgbGluZS5sb3dlcigpIGluIHZhbGlkX2xpbmVzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBzIGluIHNlbnRlbmNlX3NwbGl0KGxpbmUpOgogICAgICAgICAgICBzdGF0c1sic2VudGVuY2VzX3NjYW5uZWQiXSArPSAxCiAgICAgICAgICAgIGlmIGxlbihzLnNwbGl0KCkpIDwgNiBvciBsZW4ocy5zcGxpdCgpKSA+IDQwOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgcmUuc2VhcmNoKHIiXGR7Myx9Iiwgcykgb3IgJyInIGluIHMgb3IgIigiIGluIHM6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBmb3IgcSwgYSBpbiBkZWZpbml0aW9uYWxfcGFpcnMocywgZnJlcSk6CiAgICAgICAgICAgICAgICBpZiAocSwgYSkgaW4gc2VlbjoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgc2Vlbi5hZGQoKHEsIGEpKTsgcGFpcnMuYXBwZW5kKChxLCBhKSkKICAgICAgICAgICAgICAgIHN0YXRzWyJkZWZpbml0aW9uYWwiXSArPSAxCiAgICAgICAgICAgIGZvciBxLCBhIGluIHllc19ub19wYWlycyhzKToKICAgICAgICAgICAgICAgIGlmIChxLCBhKSBpbiBzZWVuOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBzZWVuLmFkZCgocSwgYSkpOyBwYWlycy5hcHBlbmQoKHEsIGEpKQogICAgICAgICAgICAgICAgc3RhdHNbInllc19ubyJdICs9IDEKICAgICAgICAgICAgZm9yIHEsIGEgaW4gb25lX3dvcmRfcGFpcnMocyk6CiAgICAgICAgICAgICAgICBpZiAocSwgYSkgaW4gc2VlbjoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgc2Vlbi5hZGQoKHEsIGEpKTsgcGFpcnMuYXBwZW5kKChxLCBhKSkKICAgICAgICAgICAgICAgIHN0YXRzWyJvbmVfd29yZCJdICs9IDEKCiAgICBmb3IgcSwgYSBpbiBsaXN0X3BhaXJzKHJuZyk6CiAgICAgICAgaWYgKHEsIGEpIG5vdCBpbiBzZWVuOgogICAgICAgICAgICBzZWVuLmFkZCgocSwgYSkpOyBwYWlycy5hcHBlbmQoKHEsIGEpKQogICAgICAgICAgICBzdGF0c1sibGlzdF90aHJlZSJdICs9IDEKCiAgICBmb3IgcSwgYSBpbiBvbmVfd29yZF9uZWdhdGl2ZV9wYWlycyhybmcpOgogICAgICAgIGlmIChxLCBhKSBub3QgaW4gc2VlbjoKICAgICAgICAgICAgc2Vlbi5hZGQoKHEsIGEpKTsgcGFpcnMuYXBwZW5kKChxLCBhKSkKICAgICAgICAgICAgc3RhdHNbInllc19ubyJdICs9IDEKCiAgICBybmcuc2h1ZmZsZShwYWlycykKCiAgICBuX3ZhbGlkID0gbWF4KDUwLCBpbnQobGVuKHBhaXJzKSAqIFZBTElEX0ZSQUNUSU9OKSkKICAgIHZhbGlkX3BhaXJzLCB0cmFpbl9wYWlycyA9IHBhaXJzWzpuX3ZhbGlkXSwgcGFpcnNbbl92YWxpZDpdCgogICAgT1VUX0RJUi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0ciA9IE9VVF9ESVIgLyAicWFfcGFpcnNfdHJhaW4uanNvbmwiCiAgICB2YSA9IE9VVF9ESVIgLyAicWFfcGFpcnNfdmFsaWQuanNvbmwiCiAgICB0ci53cml0ZV90ZXh0KCJcbiIuam9pbihqc29uLmR1bXBzKHsicSI6IHEsICJhIjogYX0pCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgcSwgYSBpbiB0cmFpbl9wYWlycykgKyAiXG4iLAogICAgICAgICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiKQogICAgdmEud3JpdGVfdGV4dCgiXG4iLmpvaW4oanNvbi5kdW1wcyh7InEiOiBxLCAiYSI6IGF9KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIHEsIGEgaW4gdmFsaWRfcGFpcnMpICsgIlxuIiwKICAgICAgICAgICAgICAgICAgZW5jb2Rpbmc9InV0Zi04IikKCiAgICBzdGF0cy51cGRhdGUoeyJuX3BhaXJzX3RvdGFsIjogbGVuKHBhaXJzKSwgIm5fdHJhaW4iOiBsZW4odHJhaW5fcGFpcnMpLAogICAgICAgICAgICAgICAgICAibl92YWxpZCI6IGxlbih2YWxpZF9wYWlycyksICJtaW5fc3ViamVjdF9mcmVxIjogTUlOX0ZSRVEsCiAgICAgICAgICAgICAgICAgICJzb3VyY2VfdHJhaW4iOiBzdHIoVjRfVFJBSU4ucmVsYXRpdmVfdG8oUk9PVCkpLAogICAgICAgICAgICAgICAgICAic291cmNlX3ZhbGlkX2V4Y2x1ZGVkIjogc3RyKFY0X1ZBTElELnJlbGF0aXZlX3RvKFJPT1QpKX0pCiAgICAoT1VUX0RJUiAvICJxYV9zdGF0cy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICBqc29uLmR1bXBzKHN0YXRzLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgcHJpbnQoIlFBIFBBSVIgRVhUUkFDVElPTiB2MiDigJQgY29uc3RyYWludCBmb3JtYXRzICsgcXVhbGl0eSBmaXhlcyIpCiAgICBmb3IgaywgdiBpbiBzdGF0cy5pdGVtcygpOgogICAgICAgIHByaW50KGYiICB7a306IHt2Oix9IiBpZiBpc2luc3RhbmNlKHYsIGludCkgZWxzZSBmIiAge2t9OiB7dn0iKQogICAgcHJpbnQoIlxuZXhhbXBsZXMgcGVyIGZvcm1hdDoiKQogICAgc2hvd24gPSBzZXQoKQogICAgZm9yIHEsIGEgaW4gdHJhaW5fcGFpcnM6CiAgICAgICAgZm10ID0gKCJ5ZXMvbm8iIGlmICJ5ZXMgb3Igbm8iIGluIHEgZWxzZQogICAgICAgICAgICAgICAib25lLXdvcmQiIGlmICJzaW5nbGUgd29yZCIgaW4gcSBlbHNlCiAgICAgICAgICAgICAgICJsaXN0LXRocmVlIiBpZiAidGhyZWUiIGluIHEgZWxzZSAiZGVmaW5pdGlvbmFsIikKICAgICAgICBpZiBmbXQgbm90IGluIHNob3duOgogICAgICAgICAgICBzaG93bi5hZGQoZm10KQogICAgICAgICAgICBwcmludChmIiAgW3tmbXR9XSBROiB7cX0gIEE6IHthfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='''
B64_TRAIN_QA = '''IiIiUUEgZmluZS10dW5pbmcgZm9yIEtyaXNoaUdQVC1uYW5vICh2NCBpbnN0cnVjdGlvbiBwaGFzZSkuCgpUcmFpbnMgdGhlIHdhcm0tc3RhcnRlZCBtb2RlbCBvbiBhdXRvLWdlbmVyYXRlZCBRQSBwYWlycyAoZGF0YS9xYS8pIGluIHRoZQpwbGFpbiAnUTogLi4uIEE6IC4uLicgdGV4dCBmb3JtYXQgdGhlIGNvcnB1cyBCUEUgYWxyZWFkeSBoYW5kbGVzLiBUaGlzIGlzCnN0YW5kYXJkIExNIGZpbmUtdHVuaW5nIChDRSBvbiBzaGlmdGVkIHRhcmdldHMpIOKAlCBubyBhcmNoaXRlY3R1cmUsIHRva2VuaXplciwKb3IgbG9zcyBjaGFuZ2VzLgoKRGVzaWduOgogIC0gcGFpcnMgYXJlIHNlcmlhbGl6ZWQgYXM6ICAiUTogPHF1ZXN0aW9uPlxuQTogPGFuc3dlcj4iICsgbmV3bGluZSBFT1MgZG9jCiAgICBzZXBhcmF0b3IsIGNvbmNhdGVuYXRlZCBpbnRvIG9uZSB0b2tlbiBzdHJlYW0sIHdpbmRvd2VkIGxpa2UgcHJldHJhaW5pbmcKICAtIHZlcnkgc21hbGwgTFIgKGRlZmF1bHQgMWUtNCBwZWFrLCBjb3NpbmUgdG8gMWUtNSk6IHRoZSBnb2FsIGlzIEZPUk1BVAogICAgbGVhcm5pbmcsIG5vdCBrbm93bGVkZ2UgaW5qZWN0aW9uICg2MTYgcGFpcnMgY2Fubm90IHRlYWNoIGZhY3RzKQogIC0gdmFsaWRhdGlvbiA9IGhlbGQtb3V0IFFBIHBhaXJzICg1MCkgKyB0aGUgYnl0ZS1maXhlZCBjb3JwdXMgdmFsIHNldCwgYm90aAogICAgbG9nZ2VkOyBiZXN0LnB0IHRyYWNrZWQgb24gUUEtdmFsaWQgbG9zcwogIC0gcmVzdW1lLXNhZmU6IHN0ZXAgY2hlY2twb2ludHMgKyBmaW5kX2xhdGVzdCwgc2FtZSBhcyBwcmV0cmFpbmluZwoKUnVuOiAgcHl0aG9uIHRyYWluaW5nL3RyYWluX3FhLnB5IC0tY29uZmlnIGNvbmZpZ3MvbmFub19hZ3JpX3Y0X3FhLmpzb24KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKCmZyb20gZGF0YS5kYXRhc2V0IGltcG9ydCBTZXF1ZW5jZURhdGFzZXQsIGxvYWRfaWRfc3RyZWFtICAjIG5vcWE6IEU0MDIKZnJvbSBtb2RlbC5ncHQgaW1wb3J0IEdQVCwgR1BUQ29uZmlnICAgICAgICAgICAgICAgICAgICAgICAgIyBub3FhOiBFNDAyCmZyb20gdHJhaW5pbmcuY2hlY2twb2ludCBpbXBvcnQgKCAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIG5vcWE6IEU0MDIKICAgIGZpbmRfbGF0ZXN0LAogICAgbG9hZF9jaGVja3BvaW50LAogICAgc2F2ZV9jaGVja3BvaW50LAopCmZyb20gdHJhaW5pbmcubG9zcyBpbXBvcnQgbmV4dF90b2tlbl9jZSwgcGVycGxleGl0eSAgICAgICAgICAjIG5vcWE6IEU0MDIKZnJvbSB0cmFpbmluZy50cmFpbiBpbXBvcnQgYnVpbGRfcGFyYW1fZ3JvdXBzLCBldmFsdWF0ZSAgICAgICMgbm9xYTogRTQwMgpmcm9tIHRva2VuaXplciBpbXBvcnQgQlBFVG9rZW5pemVyICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBub3FhOiBFNDAyCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0KCgpkZWYgcWFfdG9fdGV4dChwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBwYXJ0cyA9IFtdCiAgICBmb3IgbGluZSBpbiBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zcGxpdGxpbmVzKCk6CiAgICAgICAgaWYgbGluZS5zdHJpcCgpOgogICAgICAgICAgICBkID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZiJROiB7ZFsncSddfVxuQToge2RbJ2EnXX0iKQogICAgcmV0dXJuICJcblxuIi5qb2luKHBhcnRzKSArICJcbiIKCgpkZWYgZW5jb2RlX2NhY2hlZCh0b2s6IEJQRVRva2VuaXplciwgdGV4dDogc3RyLCBjYWNoZV9wYXRoOiBQYXRoKSAtPiB0b3JjaC5UZW5zb3I6CiAgICBpZiBjYWNoZV9wYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybiB0b3JjaC5sb2FkKGNhY2hlX3BhdGgpCiAgICBmcm9tIHRva2VuaXplci53b3JkX3Rva2VuaXplciBpbXBvcnQgd29yZF90b2tlbml6ZQogICAgY2FjaGU6IGRpY3Rbc3RyLCBsaXN0W2ludF1dID0ge30KICAgIGlkczogbGlzdFtpbnRdID0gW10KICAgIGZvciB3b3JkIGluIHdvcmRfdG9rZW5pemUodGV4dCk6CiAgICAgICAgaGl0ID0gY2FjaGUuZ2V0KHdvcmQpCiAgICAgICAgaWYgaGl0IGlzIE5vbmU6CiAgICAgICAgICAgIGhpdCA9IGNhY2hlW3dvcmRdID0gdG9rLmVuY29kZSh3b3JkKQogICAgICAgIGlkcy5leHRlbmQoaGl0KQogICAgdCA9IHRvcmNoLnRlbnNvcihpZHMsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBjYWNoZV9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0b3JjaC5zYXZlKHQsIGNhY2hlX3BhdGgpCiAgICByZXR1cm4gdAoKCmRlZiBnZXRfbHIoc3RlcCwgd2FybXVwLCBtYXhfbHIsIG1pbl9sciwgdG90YWwpOgogICAgaWYgc3RlcCA8IHdhcm11cDoKICAgICAgICByZXR1cm4gbWF4X2xyICogKHN0ZXAgKyAxKSAvIG1heCgxLCB3YXJtdXApCiAgICBpZiBzdGVwID49IHRvdGFsIG9yIHRvdGFsIDw9IHdhcm11cDoKICAgICAgICByZXR1cm4gbWluX2xyCiAgICBwID0gKHN0ZXAgLSB3YXJtdXApIC8gKHRvdGFsIC0gd2FybXVwKQogICAgcmV0dXJuIG1pbl9sciArIDAuNSAqIChtYXhfbHIgLSBtaW5fbHIpICogKDEgKyBtYXRoLmNvcyhtYXRoLnBpICogcCkpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKICAgIGNvbmZpZyA9IGpzb24ubG9hZHMoUGF0aChhcmdzLmNvbmZpZykucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQoKICAgIGRhdGFfcm9vdCA9IFJPT1QgLyAiZGF0YSIKICAgIHFhX2RpciA9IGRhdGFfcm9vdCAvIGNvbmZpZy5nZXQoInFhX2RpciIsICJxYSIpCiAgICBydW5fbmFtZSA9IGNvbmZpZ1sicnVuX25hbWUiXQogICAgY2twdF9kaXIgPSBST09UIC8gImNoZWNrcG9pbnRzIiAvIHJ1bl9uYW1lCiAgICBja3B0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBsb2dfcGF0aCA9IGNrcHRfZGlyIC8gInRyYWluX2xvZy5qc29ubCIKCiAgICBkZXZpY2UgPSBjb25maWcuZ2V0KCJkZXZpY2UiLCAiY3B1IikKICAgIHRvcmNoLm1hbnVhbF9zZWVkKGNvbmZpZ1sic2VlZCJdKQogICAgaWYgZGV2aWNlID09ICJjcHUiOgogICAgICAgIHRvcmNoLnNldF9udW1fdGhyZWFkcyhjb25maWcuZ2V0KCJ0b3JjaF90aHJlYWRzIiwgMTApKQoKICAgIHRvayA9IEJQRVRva2VuaXplci5sb2FkKGRhdGFfcm9vdCAvICJwcm9jZXNzZWQiIC8gImFncmlfYnBlX3Rva2VuaXplci5qc29uIikKICAgIHRyYWluX2lkcyA9IGVuY29kZV9jYWNoZWQodG9rLCBxYV90b190ZXh0KHFhX2RpciAvICJxYV9wYWlyc190cmFpbi5qc29ubCIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBxYV9kaXIgLyAicWFfdHJhaW5faWRzLnB0IikKICAgIHZhbGlkX2lkcyA9IGVuY29kZV9jYWNoZWQodG9rLCBxYV90b190ZXh0KHFhX2RpciAvICJxYV9wYWlyc192YWxpZC5qc29ubCIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBxYV9kaXIgLyAicWFfdmFsaWRfaWRzLnB0IikKCiAgICBzZXEgPSBjb25maWdbInNlcV9sZW4iXQogICAgdHJhaW5fZHMgPSBTZXF1ZW5jZURhdGFzZXQodHJhaW5faWRzLCBzZXEpCiAgICB2YWxpZF9kcyA9IFNlcXVlbmNlRGF0YXNldCh2YWxpZF9pZHMsIHNlcSkKICAgIGcgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChjb25maWdbInNlZWQiXSkKICAgIHRyYWluX2RsID0gdG9yY2gudXRpbHMuZGF0YS5EYXRhTG9hZGVyKAogICAgICAgIHRyYWluX2RzLCBiYXRjaF9zaXplPWNvbmZpZ1siYmF0Y2hfc2l6ZSJdLCBzaHVmZmxlPVRydWUsCiAgICAgICAgZHJvcF9sYXN0PVRydWUsIGdlbmVyYXRvcj1nLCBudW1fd29ya2Vycz0wKQogICAgdmFsaWRfZGwgPSB0b3JjaC51dGlscy5kYXRhLkRhdGFMb2FkZXIoCiAgICAgICAgdmFsaWRfZHMsIGJhdGNoX3NpemU9Y29uZmlnWyJiYXRjaF9zaXplIl0sIHNodWZmbGU9RmFsc2UsCiAgICAgICAgZHJvcF9sYXN0PUZhbHNlLCBudW1fd29ya2Vycz0wKQoKICAgICMgY29ycHVzIGJ5dGUtZml4ZWQgdmFsaWRhdGlvbiBmb3IgTE0tcmVncmVzc2lvbiBtb25pdG9yaW5nCiAgICBjb3JwdXNfdmFsaWRfdHh0ID0gZGF0YV9yb290IC8gImNvcnB1c192NCIgLyAiYWdyaV92YWxpZF92NC50eHQiCiAgICBjb3JwdXNfdmFsaWRfaWRzID0gbG9hZF9pZF9zdHJlYW0oCiAgICAgICAgY29ycHVzX3ZhbGlkX3R4dCwgZGF0YV9yb290IC8gImNvcnB1c192NCIgLyAiYWdyaV92YWxpZF9pZHMucHQiLCB0b2spCiAgICBjb3JwdXNfdmFsaWRfZGwgPSB0b3JjaC51dGlscy5kYXRhLkRhdGFMb2FkZXIoCiAgICAgICAgU2VxdWVuY2VEYXRhc2V0KGNvcnB1c192YWxpZF9pZHMsIHNlcSksIGJhdGNoX3NpemU9Y29uZmlnWyJiYXRjaF9zaXplIl0sCiAgICAgICAgc2h1ZmZsZT1GYWxzZSwgbnVtX3dvcmtlcnM9MCkKCiAgICBtYXhfc3RlcHMgPSBjb25maWdbIm1heF9zdGVwcyJdCiAgICBjZmcgPSBHUFRDb25maWcodm9jYWJfc2l6ZT10b2sudm9jYWJfc2l6ZSwgbWF4X2xlbj1zZXEsCiAgICAgICAgICAgICAgICAgICAgZHJvcG91dD1jb25maWdbImRyb3BvdXQiXSkKICAgIG1vZGVsID0gR1BUKGNmZykudG8oZGV2aWNlKQogICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbVcoYnVpbGRfcGFyYW1fZ3JvdXBzKG1vZGVsLCBjb25maWdbIndlaWdodF9kZWNheSJdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxyPWNvbmZpZ1sibHIiXSwgYmV0YXM9dHVwbGUoY29uZmlnWyJiZXRhcyJdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVwcz1jb25maWdbImVwcyJdKQoKICAgIHdhcm1fZmlsZSA9IFJPT1QgLyBjb25maWdbIndhcm1fc3RhcnQiXSBpZiBjb25maWcuZ2V0KAogICAgICAgICJ3YXJtX3N0YXJ0IikgZWxzZSBOb25lCiAgICBsYXRlc3QgPSBmaW5kX2xhdGVzdChja3B0X2RpcikKICAgIHN0YXJ0X3N0ZXAsIGJlc3RfcWFfbG9zcywgdG9rZW5zX3NlZW4gPSAwLCBmbG9hdCgiaW5mIiksIDAKICAgIGlmIGxhdGVzdCBpcyBub3QgTm9uZToKICAgICAgICBzdGF0ZSA9IGxvYWRfY2hlY2twb2ludChsYXRlc3QsIG1vZGVsLCBvcHQpCiAgICAgICAgc3RhcnRfc3RlcCA9IHN0YXRlWyJzdGVwIl0KICAgICAgICB0b2tlbnNfc2VlbiA9IHN0YXRlWyJ0b2tlbnNfc2VlbiJdCiAgICAgICAgYmVzdF9xYV9sb3NzID0gc3RhdGVbImJlc3RfdmFsX2xvc3MiXQogICAgICAgIHByaW50KGYicmVzdW1lZCBmcm9tIHtsYXRlc3QubmFtZX06IHN0ZXAge3N0YXJ0X3N0ZXB9IikKICAgIGVsaWYgd2FybV9maWxlIGlzIG5vdCBOb25lOgogICAgICAgIGxvYWRfY2hlY2twb2ludCh3YXJtX2ZpbGUsIG1vZGVsLCBvcHQpCiAgICAgICAgcHJpbnQoZiJ3YXJtIHN0YXJ0IGZyb20ge3dhcm1fZmlsZS5uYW1lfTogd2VpZ2h0cyArIG9wdGltaXplciBsb2FkZWQiKQoKICAgIG5fcGFyYW1zID0gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkpCiAgICBwcmludChmIm1vZGVsOiB7bl9wYXJhbXM6LH0gcGFyYW1zIHwgUUEgdHJhaW4gd2luZG93cyB7bGVuKHRyYWluX2RzKTosfSAiCiAgICAgICAgICBmIih7bGVuKHRyYWluX2RsKX0gc3RlcHMvZXBvY2gpIHwgUUEgdmFsaWQgd2luZG93cyB7bGVuKHZhbGlkX2RzKTosfSIpCgogICAgc3RlcCA9IHN0YXJ0X3N0ZXAKICAgIHRyYWluX2l0ZXIgPSBpdGVyKHRyYWluX2RsKQogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgd2hpbGUgc3RlcCA8IG1heF9zdGVwczoKICAgICAgICBsciA9IGdldF9scihzdGVwLCBjb25maWdbIndhcm11cF9zdGVwcyJdLCBjb25maWdbImxyIl0sCiAgICAgICAgICAgICAgICAgICAgY29uZmlnWyJtaW5fbHIiXSwgbWF4X3N0ZXBzKQogICAgICAgIGZvciBncm91cCBpbiBvcHQucGFyYW1fZ3JvdXBzOgogICAgICAgICAgICBncm91cFsibHIiXSA9IGxyCiAgICAgICAgdHJ5OgogICAgICAgICAgICB4YiwgeWIgPSBuZXh0KHRyYWluX2l0ZXIpCiAgICAgICAgZXhjZXB0IFN0b3BJdGVyYXRpb246CiAgICAgICAgICAgIHRyYWluX2l0ZXIgPSBpdGVyKHRyYWluX2RsKQogICAgICAgICAgICB4YiwgeWIgPSBuZXh0KHRyYWluX2l0ZXIpCiAgICAgICAgb3B0Lnplcm9fZ3JhZCgpCiAgICAgICAgbG9zcyA9IG5leHRfdG9rZW5fY2UobW9kZWwoeGIudG8oZGV2aWNlKSksIHliLnRvKGRldmljZSkpCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgY29uZmlnWyJncmFkX2NsaXAiXSkKICAgICAgICBvcHQuc3RlcCgpCiAgICAgICAgc3RlcCArPSAxCiAgICAgICAgdG9rZW5zX3NlZW4gKz0geGIubnVtZWwoKQoKICAgICAgICBpZiBzdGVwICUgY29uZmlnWyJsb2dfZXZlcnkiXSA9PSAwOgogICAgICAgICAgICBkdCA9IHRpbWUudGltZSgpIC0gdDAKICAgICAgICAgICAgdHBzID0geGIubnVtZWwoKSAvIG1heChkdCwgMWUtOSkKICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICBlbnRyeSA9IHsic3RlcCI6IHN0ZXAsICJsciI6IHJvdW5kKGxyLCA4KSwKICAgICAgICAgICAgICAgICAgICAgInRyYWluX2xvc3MiOiByb3VuZChsb3NzLml0ZW0oKSwgNCksCiAgICAgICAgICAgICAgICAgICAgICJ0cmFpbl9wcGwiOiByb3VuZChwZXJwbGV4aXR5KGxvc3MuaXRlbSgpKSwgMiksCiAgICAgICAgICAgICAgICAgICAgICJ0b2tlbnNfc2VlbiI6IHRva2Vuc19zZWVuLAogICAgICAgICAgICAgICAgICAgICAidG9rZW5zX3Blcl9zZWMiOiByb3VuZCh0cHMsIDEpfQogICAgICAgICAgICB3aXRoIG9wZW4obG9nX3BhdGgsICJhIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyhlbnRyeSkgKyAiXG4iKQogICAgICAgICAgICBwcmludChmInN0ZXAge3N0ZXA6PjV9IHwgcWFfbG9zcyB7bG9zcy5pdGVtKCk6LjRmfSAiCiAgICAgICAgICAgICAgICAgIGYicHBsIHtwZXJwbGV4aXR5KGxvc3MuaXRlbSgpKTouMmZ9IHwgbHIge2xyOi4yZX0iKQoKICAgICAgICBpZiBzdGVwICUgY29uZmlnWyJ2YWxfZXZlcnkiXSA9PSAwOgogICAgICAgICAgICBtb2RlbC5ldmFsKCkKICAgICAgICAgICAgcWFfdmFsID0gZXZhbHVhdGUobW9kZWwsIHZhbGlkX2RsLCBkZXZpY2UpCiAgICAgICAgICAgIGNvcnB1c192YWwgPSBldmFsdWF0ZShtb2RlbCwgY29ycHVzX3ZhbGlkX2RsLCBkZXZpY2UpCiAgICAgICAgICAgIG1vZGVsLnRyYWluKCkKICAgICAgICAgICAgaW1wcm92ZWQgPSBxYV92YWwgPCBiZXN0X3FhX2xvc3MKICAgICAgICAgICAgaWYgaW1wcm92ZWQ6CiAgICAgICAgICAgICAgICBiZXN0X3FhX2xvc3MgPSBxYV92YWwKICAgICAgICAgICAgICAgIHNhdmVfY2hlY2twb2ludChja3B0X2RpciAvICJiZXN0LnB0IiwgbW9kZWwsIG9wdCwgc3RlcCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b2tlbnNfc2VlbiwgYmVzdF9xYV9sb3NzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsqKmNvbmZpZywgInZvY2FiX3NpemUiOiB0b2sudm9jYWJfc2l6ZX0sIGxyKQogICAgICAgICAgICB3aXRoIG9wZW4obG9nX3BhdGgsICJhIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyh7CiAgICAgICAgICAgICAgICAgICAgInN0ZXAiOiBzdGVwLCAicWFfdmFsX2xvc3MiOiByb3VuZChxYV92YWwsIDQpLAogICAgICAgICAgICAgICAgICAgICJxYV92YWxfcHBsIjogcm91bmQocGVycGxleGl0eShxYV92YWwpLCAyKSwKICAgICAgICAgICAgICAgICAgICAiY29ycHVzX3ZhbF9sb3NzIjogcm91bmQoY29ycHVzX3ZhbCwgNCksCiAgICAgICAgICAgICAgICAgICAgImNvcnB1c192YWxfcHBsIjogcm91bmQocGVycGxleGl0eShjb3JwdXNfdmFsKSwgMiksCiAgICAgICAgICAgICAgICAgICAgImJlc3QiOiBpbXByb3ZlZH0pICsgIlxuIikKICAgICAgICAgICAgcHJpbnQoZiIgIFZBTElEQVRJT04gc3RlcCB7c3RlcH06IHFhX3ZhbCB7cWFfdmFsOi40Zn0gIgogICAgICAgICAgICAgICAgICBmIihwcGwge3BlcnBsZXhpdHkocWFfdmFsKTouMmZ9KSB8IGNvcnB1c192YWwgIgogICAgICAgICAgICAgICAgICBmIntjb3JwdXNfdmFsOi40Zn0gKHBwbCB7cGVycGxleGl0eShjb3JwdXNfdmFsKTouMmZ9KSIKICAgICAgICAgICAgICAgICAgZiJ7JyAgKmJlc3QqJyBpZiBpbXByb3ZlZCBlbHNlICcnfSIpCgogICAgICAgIGlmIChzdGVwIC0gc3RhcnRfc3RlcCkgYW5kIHN0ZXAgJSBjb25maWdbInNhdmVfZXZlcnkiXSA9PSAwOgogICAgICAgICAgICBzYXZlX2NoZWNrcG9pbnQoY2twdF9kaXIgLyBmInN0ZXBfe3N0ZXA6MDdkfS5wdCIsIG1vZGVsLCBvcHQsIHN0ZXAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b2tlbnNfc2VlbiwgYmVzdF9xYV9sb3NzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgeyoqY29uZmlnLCAidm9jYWJfc2l6ZSI6IHRvay52b2NhYl9zaXplfSwgbHIpCgogICAgZmluYWxfbHIgPSBnZXRfbHIoc3RlcCwgY29uZmlnWyJ3YXJtdXBfc3RlcHMiXSwgY29uZmlnWyJsciJdLAogICAgICAgICAgICAgICAgICAgICAgY29uZmlnWyJtaW5fbHIiXSwgbWF4X3N0ZXBzKQogICAgc2F2ZV9jaGVja3BvaW50KGNrcHRfZGlyIC8gImZpbmFsLnB0IiwgbW9kZWwsIG9wdCwgc3RlcCwgdG9rZW5zX3NlZW4sCiAgICAgICAgICAgICAgICAgICAgYmVzdF9xYV9sb3NzLCB7Kipjb25maWcsICJ2b2NhYl9zaXplIjogdG9rLnZvY2FiX3NpemV9LAogICAgICAgICAgICAgICAgICAgIGxyPWZpbmFsX2xyKQogICAgcHJpbnQoZiJcbkRPTkUgYXQgc3RlcCB7c3RlcH06IGJlc3QgUUEgdmFsIGxvc3Mge2Jlc3RfcWFfbG9zczouNGZ9OyAiCiAgICAgICAgICBmImNoZWNrcG9pbnRzIGluIHtja3B0X2Rpcn0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'''

gen_dir = WORK / 'evaluation' / 'corpus_v4'
gen_dir.mkdir(parents=True, exist_ok=True)
(gen_dir / 'make_qa_pairs.py').write_text(
    base64.b64decode(B64_MAKE_QA_PAIRS).decode('utf-8'), encoding='utf-8')
(gen_dir / 'make_qa_pairs_v2.py').write_text(
    base64.b64decode(B64_MAKE_QA_PAIRS_V2).decode('utf-8'), encoding='utf-8')
(WORK / 'training' / 'train_qa.py').write_text(
    base64.b64decode(B64_TRAIN_QA).decode('utf-8'), encoding='utf-8')  # qa_dir key included
print('generator + trainer scripts written (byte-identical to the '
      'locally-verified copies; trainer now supports qa_dir)')

# ---- generate data/qa_v2 from the corpus already on Drive -------------------
r = subprocess.run([sys.executable, 'evaluation/corpus_v4/make_qa_pairs_v2.py'],
                   cwd=str(WORK))
assert r.returncode == 0, 'pair generation failed — see log above'

stats = json.loads((WORK / 'data' / 'qa_v2' / 'qa_stats.json').read_text())
print('pair stats:', {k: stats[k] for k in
                      ('n_pairs_total', 'n_train', 'n_valid')})
assert (stats['n_pairs_total'], stats['n_train'], stats['n_valid']) == \
       (851, 801, 50), (
    'pair counts differ from the verified local generation (851/801/50) — '
    'the Drive corpus_v4 text may differ from the local one. STOP and '
    'paste this output back before training.')

# drop any stale id caches so the trainer re-tokenizes the new data
for f in (WORK / 'data' / 'qa_v2').glob('*.pt'):
    f.unlink()

# ---- write the round-2 config with the runtime device ------------------------
import torch
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print('device:', torch.cuda.get_device_name(0))
else:
    print('no GPU — CPU fallback (much slower)')
cfg = json.loads('''{"run_name": "krishigpt_v4_qa2", "qa_dir": "qa_v2", "device": "cpu", "torch_threads": 10, "seed": 0, "seq_len": 128, "batch_size": 8, "epochs": 3, "max_steps": 63, "lr": 0.0001, "min_lr": 1e-05, "warmup_steps": 10, "weight_decay": 0.01, "grad_clip": 1.0, "betas": [0.9, 0.95], "eps": 1e-08, "dropout": 0.0, "log_every": 5, "val_every": 21, "save_every": 21, "warm_start": "checkpoints/krishigpt_v4_qa/best.pt", "notes": "ROUND 2 QA fine-tune on data/qa_v2 (801 train / 50 valid pairs; trainer-measured 174 seq-128 windows = 21 steps/epoch at bs 8 \u2014 same epoch length as round 1 because the 369 short constraint answers offset the extra pairs). Fixes round-1 causes: (1) constraint formats now demonstrated in data \u2014 yes/no (369, balanced 251 yes/118 no), one-word (18), list-of-three (15), alongside 399 definitional; (2) subject extractor requires sentence-initial domain-term heads (0 bad subjects vs round-1's 'whatever soil'/'as each cow' pairs). BENCHMARK OVERFIT GUARD: training phrasing ('Answer yes or no.'/'Reply with a single word.'/'Name three ...') is deliberately different from the frozen 10-test bench phrasing; the <=10-words, JSON, and stop-word formats are never trained. Schedule: 3 real epochs = 63 steps (21/epoch), val/save every epoch, LR 1e-4 -> 1e-5, warm start from krishigpt_v4_qa/best.pt (round-1 format-tuned weights \u2014 surface Q/A format already known, so this round spends its budget on constraint shapes). Monitors held-out QA-valid loss (best.pt criterion) and byte-fixed corpus val loss (regression guard vs 3.9752)."}''')
cfg['device'] = DEVICE
if DEVICE == 'cpu':
    cfg['torch_threads'] = 8
(WORK / 'configs' / 'nano_agri_v4_qa2.json').write_text(
    json.dumps(cfg, indent=2))
print(f"config written: device={DEVICE} | max_steps={cfg['max_steps']} "
      f"(3 epochs x 21 steps) | warm_start={cfg['warm_start']}")


In [ ]:
# 4. ROUND 2 — constraint-format QA fine-tune (63 steps, ~2-3 min on T4).
# Restart-safe: re-run after a disconnect to resume from the latest step_*.pt.
# Fresh-run semantics: if a previous qa2 dir exists, archive best.pt and
# clear resume points so training starts clean from the round-1 donor.
import shutil
from pathlib import Path

qa2_dir = WORK / 'checkpoints' / 'krishigpt_v4_qa2'
if qa2_dir.exists():
    for f in qa2_dir.iterdir():
        if f.name.startswith('step_') or f.name in ('final.pt',):
            f.unlink()
    for f in qa2_dir.glob('*.tmp'):
        f.unlink()
    old = qa2_dir / 'best.pt'
    if old.exists():
        old.rename(qa2_dir / 'best_prev.pt')
    (qa2_dir / 'train_log.jsonl').unlink(missing_ok=True)
    print('cleared previous qa2 run dir (old best kept as best_prev.pt)')

import subprocess, sys
r = subprocess.run([sys.executable, 'training/train_qa.py',
                    '--config', 'configs/nano_agri_v4_qa2.json'],
                   cwd=str(WORK))
assert r.returncode == 0, 'round-2 QA tuning failed — see log above'

# verify the run actually trained on qa_v2 with the corrected schedule:
# best.pt carries the run config; check the distinctive fields
import torch as _torch
state = _torch.load(WORK / 'checkpoints' / 'krishigpt_v4_qa2' / 'best.pt',
                    map_location='cpu', weights_only=False)
cfg_in_ckpt = state['config']
assert cfg_in_ckpt.get('qa_dir') == 'qa_v2', \
    f"trainer used qa_dir={cfg_in_ckpt.get('qa_dir')!r}, expected 'qa_v2'"
assert cfg_in_ckpt['max_steps'] == 63, cfg_in_ckpt['max_steps']
assert cfg_in_ckpt['warm_start'] == 'checkpoints/krishigpt_v4_qa/best.pt'
n_epochs_done = state['step'] / 21
print(f"verified: qa_dir=qa_v2 | donor=v4_qa best | stopped at step "
      f"{state['step']} ({n_epochs_done:.2f} epochs) | best QA val loss "
      f"{state['best_val_loss']:.4f}")


In [ ]:
# 5. FROZEN FUNNEL EVALUATION on the round-2 checkpoint.
import subprocess, sys

for script, arg in (('evaluation/health_report.py', 'krishigpt_v4_qa2'),
                    ('evaluation/instruction_bench.py', 'krishigpt_v4_qa2'),
                    ('evaluation/mcq_bench.py', 'krishigpt_v4_qa2'),
                    ('evaluation/funnel_report.py', 'krishigpt_v4_qa2')):
    print(f'== {script} {arg}')
    r = subprocess.run([sys.executable, script, arg], cwd=str(WORK))
    print()


In [ ]:
# 6. Three-way comparison: v3 (frozen baseline) vs v4_qa (round 1) vs qa2.
import json

res = lambda run, name: json.loads(
    (WORK / 'evaluation' / 'results' / run / name).read_text())

for name in ('instruction_bench.json', 'mcq_bench.json'):
    print(f'== {name}')
    for run in ('krishigpt_v3', 'krishigpt_v4_qa', 'krishigpt_v4_qa2'):
        try:
            d = res(run, name)
            if name.startswith('instruction'):
                print(f'  {run}: compliance {d["compliance"]:.0%}')
            else:
                print(f'  {run}: MCQ acc {d["overall_accuracy"]:.1%} '
                      f'(weighted {d["domain_score_weighted"]:.1%})')
        except FileNotFoundError:
            print(f'  {run}: (missing)')

# per-test compliance detail for round 2
d = res('krishigpt_v4_qa2', 'instruction_bench.json')
print('\nround-2 per-test:')
for t in d['results']:
    mark = 'PASS' if t['compliant'] else 'FAIL'
    print(f'  [{mark}] {t["id"]:<14} out: {t["output"][:60]!r}')


In [ ]:
# 7. Smoke-test constraint formats from the round-2 model.
import sys
sys.path.insert(0, str(WORK))
from model.generate import KrishiGenerator

gen = KrishiGenerator.from_checkpoint(
    WORK / 'checkpoints' / 'krishigpt_v4_qa2' / 'best.pt',
    WORK / 'data' / 'processed' / 'agri_bpe_tokenizer.json')
prompts = (
    'Q: Does rice need water? Answer yes or no.\nA:',
    'Q: What is loam? Reply with a single word.\nA:',
    'Q: Name three farm practices, separated by commas.\nA:',
    'Q: What is compost?\nA:',
)
for p in prompts:
    r = gen.sample(p, max_tokens=32, top_p=0.9, seed=0)
    print(f'{p}\n-> {r.text[:160]}\n')


In [ ]:
# 8. Sync round-2 artifacts back to Drive (v3 untouched), final verify.
import hashlib, shutil
from pathlib import Path

run = 'krishigpt_v4_qa2'
src = WORK / 'checkpoints' / run
dst = SRC / 'checkpoints' / run
if src.exists():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'synced {run} checkpoints -> Drive')
src = WORK / 'evaluation' / 'results' / run
dst = SRC / 'evaluation' / 'results' / run
if src.exists():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'synced {run} results -> Drive')
# keep the generated data on Drive too (record + no regeneration needed)
src = WORK / 'data' / 'qa_v2'
dst = SRC / 'data' / 'qa_v2'
if src.exists():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('synced data/qa_v2 -> Drive')

h = hashlib.sha256()
with open(WORK / 'checkpoints' / 'krishigpt_v3' / 'best.pt', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == ('137cc83da53dd873cbab22c82e7344c7c84882d79b'
                         '37017a37c765ff38e14536')
print('done — v3 checkpoint still untouched (post-run verify)')
